In [1]:
import nltk
from collections import Counter

nltk.download('gutenberg')

words = []

for fileid in nltk.corpus.gutenberg.fileids():
    words.extend(nltk.corpus.gutenberg.words(fileid))

words = [w.lower() for w in words if w.isalpha()]
word_freq = Counter(words)

def edits1(word):
    letters = 'abcdefghijklmnopqrstuvwxyz'
    splits = [(word[:i], word[i:]) for i in range(len(word)+1)]

    deletes = [L + R[1:] for L, R in splits if R]
    inserts = [L + c + R for L, R in splits for c in letters]
    replaces = [L + c + R[1:] for L, R in splits if R for c in letters]

    return set(deletes + inserts + replaces)

def edits2(word):
    return set(e2 for e1 in edits1(word) for e2 in edits1(e1))

def known(words):
    return set(w for w in words if w in word_freq)

def correct(word):
    candidates = (
        known([word]) or
        known(edits1(word)) or
        known(edits2(word)) or
        [word]
    )
    return max(candidates, key=lambda w: word_freq[w])

print("Correction:", correct("machin"))
print("Correction:", correct("lernning"))

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.


Correction: machine
Correction: leaning


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

text = "I love machine learning and deep learning"
words = text.lower().split()
vocab = list(set(words))

word_to_ix = {w:i for i,w in enumerate(vocab)}

window_size = 2
data = []

for i in range(window_size, len(words)-window_size):
    context = [words[i-2], words[i-1], words[i+1], words[i+2]]
    target = words[i]
    data.append((context, target))

class CBOW(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs).mean(dim=0)
        out = self.linear(embeds)
        return out

model = CBOW(len(vocab), 10)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

for epoch in range(50):
    for context, target in data:
        context_ix = torch.tensor([word_to_ix[w] for w in context])
        target_ix = torch.tensor([word_to_ix[target]])

        model.zero_grad()
        output = model(context_ix)
        loss = loss_fn(output.view(1,-1), target_ix)
        loss.backward()
        optimizer.step()

print("Training Complete")

Training Complete


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

text = "I love machine learning and deep learning"
words = text.lower().split()
vocab = list(set(words))

word_to_ix = {w:i for i,w in enumerate(vocab)}

window_size = 2
data = []

for i in range(len(words)):
    for j in range(-window_size, window_size+1):
        if j != 0 and 0 <= i+j < len(words):
            data.append((words[i], words[i+j]))

class SkipGram(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, input_word):
        embed = self.embeddings(input_word)
        out = self.linear(embed)
        return out

model = SkipGram(len(vocab), 10)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

for epoch in range(50):
    for target, context in data:
        target_ix = torch.tensor([word_to_ix[target]])
        context_ix = torch.tensor([word_to_ix[context]])

        model.zero_grad()
        output = model(target_ix)
        loss = loss_fn(output, context_ix)
        loss.backward()
        optimizer.step()

print("Training Complete")

Training Complete
